# Statistical Difference Test

## Introduction
In this notebook, we will test whether there is a significant difference between the multimodal and unimodal model performance.

We will:
1. Load the metric values for both models.
2. Visualize each metric distribution.
3. Test for normality using:
 - Shapiro–Wilk Test (numerical)
 - Q–Q Plot (visual)

4. Based on normality, apply:
 - Paired t-test (if normal)
 - Wilcoxon Signed Rank Test (if not normal)
5. Summarize the results clearly.

This notebook aims to make each step easy to understand, even for those new to hypothesis testing.

## Imports
These are the necessary imports to perform our statistical test for significant difference as answer to our SOP 3.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from IPython.display import display

## Raw Metrics Data

Below are the cross evaluation results of the multimodal and unimodal models for each metric.
Each list contains values gathered across the same dataset , which makes them paired data.

In [ ]:
multimodal = {
    "accuracy":  [0.9273, 0.9461, 0.9463, 0.9391, 0.9554],
    "precision": [0.93, 0.95, 0.95, 0.94, 0.96],
    "recall":    [0.93, 0.95, 0.95, 0.94, 0.95],
    "f1":        [0.93, 0.95, 0.95, 0.94, 0.96]
}

unimodal = {
    "accuracy":  [0.8433, 0.8139, 0.8147, 0.8219, 0.8512],
    "precision": [0.86, 0.85, 0.85, 0.86, 0.88],
    "recall":    [0.85, 0.82, 0.81, 0.82, 0.85],
    "f1":        [0.84, 0.81, 0.81, 0.82, 0.85]
}

metrics = ["accuracy", "precision", "recall", "f1"]

## Visualization Function

This function prints the values of both models and plots them side-by-side.
Visualizing the data helps us see whether:
- one model consistently performs better
- the scores look normally distributed
- the values have outliers or patterns

In [ ]:
def visualize_metric(metric):
    print(f"\n====== {metric.upper()} ======\n")
    
    df = pd.DataFrame({
        "Multimodal": multimodal[metric],
        "Unimodal": unimodal[metric]
    })

    display(df)

    # Bar plot visualization
    plt.figure(figsize=(6,4))
    plt.plot(df["Multimodal"], marker='o', label="Multimodal")
    plt.plot(df["Unimodal"], marker='o', label="Unimodal")
    plt.title(f"{metric.capitalize()} Samples")
    plt.ylabel(metric.capitalize())
    plt.xlabel("Sample #")
    plt.legend()
    plt.grid(True)
    plt.show()

## Normality Test
Before choosing a statistical test, we must check if the difference scores follow a normal distribution.

This function performs:
- Shapiro–Wilk Test: Gives a p-value telling whether the data is normal.
- Q–Q Plot: Produces a visual confirmation of normality.

Interpretation:
- If p-value > 0.05 → normal
- If p-value ≤ 0.05 → not normal

In [ ]:
def test_normality(metric, alpha=0.05):
    print(f"\n+++++ Normality Test for {metric.capitalize()} +++++")

    diffs = np.array(multimodal[metric]) - np.array(unimodal[metric])

    stat, p_value = stats.shapiro(diffs)
    print(f"Shapiro-Wilk Test → W={stat:.4f}, p={p_value:.4f}")


    if p_value > alpha:
        print(f"Result: p = {p_value:.4f} > α = {alpha} → Fail to reject H₀ (normality)")
        is_normal = True
        conclusion = "follow a NORMAL distribution"
    else:
        print(f"Result: p = {p_value:.4f} ≤ α = {alpha} → Reject H₀ (normality)")
        is_normal = False
        conclusion = "DO NOT follow a normal distribution"
    
    print(f"Conclusion: Differences {conclusion}.\n")

    print("Q-Q Plot of Differences:")
    plt.figure(figsize=(5,5))
    stats.probplot(diffs, dist="norm", plot=plt)
    plt.title(f"Q-Q Plot of Differences ({metric})")
    plt.grid(True)
    plt.show()

    

    return is_normal, diffs


## Hypothesis Test

This function chooses the correct test depending on normality and evaluates:
- Paired t-test (for normal data)
- Wilcoxon Signed-Rank Test (for non-normal data)

Both tests compare whether the multimodal and unimodal results are significantly different.

Interpretation:
- If p ≤ 0.05 → significant difference
- If p > 0.05 → no significant difference

In [ ]:
def run_hypothesis_test(metric, is_normal, alpha=0.05):
    print(f"\n+++++ Hypothesis Test for {metric.capitalize()} +++++")
    print("H0: No significant difference between multimodal and unimodal.")
    print("H1: There IS a significant difference.\n")

    if is_normal:
        t_stat, p_val = stats.ttest_rel(multimodal[metric], unimodal[metric])
        test_used = "Paired t-test"
    else:
        t_stat, p_val = stats.wilcoxon(multimodal[metric], unimodal[metric])
        test_used = "Wilcoxon Signed Rank Test"

    print(f"{test_used} → statistic={t_stat:.4f}, p-value={p_val:.4f}")

    if p_val < alpha:
        print(f"Result: p = {p_val: .4f} < α = {alpha} → Reject H₀")
        conclusion = "a SIGNIFICANT difference"
    else:
        print(f"Result: p = {p_val: .4f} ≥ α = {alpha} → Fail to reject H₀")
        conclusion = "NO significant difference"
    
    print(f"Conclusion: There is {conclusion} for {metric} at α=0.05.\n")

    return {
        "metric": metric,
        "distribution": "normal" if is_normal else "not normal",
        "test_used": test_used,
        "p_value": p_val,
        "significantly different": "Yes" if p_val < 0.05 else "No"
    }


## Running Tests
This loop runs everything for each metric:
1. Visualize the data.
2. Test for normality.
3. Apply the correct statistical test.
4. Store the results for summary.

In [ ]:
results_summary = []

for metric in metrics:
    visualize_metric(metric)
    is_normal, diffs = test_normality(metric)
    result = run_hypothesis_test(metric, is_normal)
    results_summary.append(result)


## Final Summary
This table gives a clean summary of the significant difference results for:
- Accuracy
- Precision
- Recall
- F1-score

It shows:
- Whether the metric differences were normal
- Which test was used
- Whether the result was significant

In [ ]:
print("\n=========================")
print(" FINAL SUMMARY OF RESULTS")
print("=========================\n")

summary_df = pd.DataFrame(results_summary)
display(summary_df)
